<a href="https://colab.research.google.com/github/felipeandrian/universal-ai-translator/blob/main/notebook/Tradutor_Universal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Universal AI Translator & Formatter
### De qualquer idioma para um Markdown impecável

Este notebook é uma solução completa para transformar artigos complexos da web em documentos **Markdown profissionais**, traduzidos e formatados. Através da integração de APIs líderes de mercado, ele resolve o problema de tradução de conteúdos massivos que normalmente travam ou são cortados em tradutores comuns.

---

###  Objetivo
Automatizar o fluxo de **curadoria de conteúdo internacional**. O sistema não apenas traduz o texto, mas atua como um editor digital, aplicando formatação estética, negritos em conceitos-chave e organizando a estrutura para garantir uma leitura fluida e profissional.

###  Tecnologias e Recursos
Para alcançar alta fidelidade e resiliência, o projeto utiliza:

*   **Newspaper3k:** Extração inteligente de conteúdo, eliminando anúncios, menus e ruídos visuais.
*   **Azure Translator API (Microsoft):** Tradução de alta performance com detecção automática de idioma.
*   **Google Gemini 2.0:** Inteligência Generativa para o acabamento estético e estruturação do Markdown.
*   **Algoritmo de Chunking & Backoff:** Sistema exclusivo que fatia textos gigantes e gerencia limites de taxa (Erro 429), garantindo estabilidade.

---

###  Configuração das Credenciais
Para utilizar este notebook, você precisará configurar suas chaves de API:

#### 1. Microsoft Azure Translator
*   Crie uma conta no [Portal da Azure](https://portal.azure.com).
*   Crie um recurso de **Tradução** (nível gratuito F0).
*   Copie sua **CHAVE** e a **REGIÃO** (ex: `eastus`).

#### 2. Google Gemini (AI Studio)
*   Acesse o [Google AI Studio](https://aistudio.google.com/).
*   Gere uma nova chave para o modelo **Gemini 2.5 Flash**.

> ** Nota de Segurança:** Insira suas chaves nas variáveis correspondentes no código abaixo. Nunca compartilhe seu notebook publicamente com as chaves preenchidas!

---

###  Como usar este Notebook
1.  **Instalação:** Execute a primeira célula para preparar o ambiente.
2.  **Código Base:** Execute a célula que contém o script `translate.py`.
3.  **Execução:** Informe a URL do artigo desejado no comando final.
4.  **Download:** O arquivo `.md` aparecerá na pasta lateral do Colab pronto para baixar.

###  Preparação do Ambiente
Nesta etapa, instalamos as bibliotecas essenciais para que o tradutor funcione corretamente:

*   **`newspaper3k`**: Responsável por navegar na URL e extrair apenas o texto relevante do artigo (limpando menus e anúncios).
*   **`lxml[html_clean]`**: Um motor de processamento de HTML robusto necessário para a extração de texto.
*   **`requests`**: Permite que o Python se comunique com a API de Tradução da Microsoft Azure.
*   **`google-genai`**: O SDK oficial para integrar o **Gemini 2.5**, que cuida da formatação estética final do documento.

In [5]:
!pip install newspaper3k lxml[html_clean] requests google-genai

### Configuração de Segurança (Colab Secrets)
Para que o tradutor funcione de forma  segura, utilizamos o recurso **Secrets** do Google Colab. Isso evita que suas senhas fiquem expostas no código.

**Como configurar:**
1.  No menu lateral esquerdo, clique no ícone da **Chave (Secrets)** 🔑.
2.  Adicione um novo segredo com o nome `AZURE_KEY` e cole sua chave da Azure no valor.
3.  Adicione outro segredo com o nome `GEMINI_KEY` e cole sua chave do Google Gemini no valor.
4.  Ative a chave de seleção "Acesso ao Notebook" para ambos.

A célula abaixo irá ler esses valores e prepará-los para o script principal.

In [38]:
import os
from google.colab import userdata

os.environ['AZURE_KEY'] = userdata.get('AZURE_KEY')
os.environ['GEMINI_KEY'] = userdata.get('GEMINI_KEY')

print(" Chaves prontas para uso!")

 Chaves prontas para uso!


### O Motor do Tradutor (translate.py)

Esta célula contém o coração do sistema. O script foi desenhado para ser resiliente e inteligente, utilizando uma arquitetura de processamento em estágios:

#### **Como o algoritmo funciona:**

1.  **Extração Inteligente (Newspaper3k):** Diferente de um simples `copy-paste`, o script identifica o que é conteúdo real e o que é "lixo" (menus, rodapés, anúncios) na URL fornecida.
2.  **Estratégia de Chunking (Fatiamento):** APIs de tradução possuem limites de caracteres por requisição. O código fatia o texto em blocos de até 4000 caracteres, garantindo que o corte ocorra em quebras de parágrafo ou espaços para não corromper o sentido.
3.  **Resiliência com Exponential Backoff:** Ao traduzir volumes massivos, a Azure pode retornar o erro `429 (Too Many Requests)`. O script detecta isso automaticamente e aplica uma pausa que dobra a cada tentativa, garantindo que a tradução chegue ao fim sem interrupções.
4.  **Refino Estético (Gemini 2.5):** A tradução bruta da Azure é funcional, mas seca. Enviamos o resultado para o Gemini com um prompt de engenharia editorial para aplicar a formatação Markdown, destacar conceitos em negrito e ajustar a fluidez do português.

In [39]:
%%writefile translate.py
import argparse
import requests
import uuid
import time
import os
from google import genai
from newspaper import Article


# ==========================================
# 1. CREDENCIAIS AZURE TRANSLATOR
# ==========================================
TRANSLATOR_CHAVE = os.environ.get('AZURE_KEY') or "INSIRA_SUA_CHAVE_AZURE_AQUI"
TRANSLATOR_REGIAO = "eastus" #coloque sua regiao azure
TRANSLATOR_ENDPOINT = "https://api.cognitive.microsofttranslator.com"

# ==========================================
# 2. CREDENCIAIS GOOGLE GEMINI
# ==========================================
GEMINI_CHAVE = os.environ.get('GEMINI_KEY') or "INSIRA_SUA_CHAVE_GEMINI_AQUI"
client = genai.Client(api_key=GEMINI_CHAVE)

# ==========================================
# FUNÇÕES DO SISTEMA
# ==========================================

def extrair_artigo_da_url(url):
    print(f"Lendo o link: {url}...")
    try:
        artigo_web = Article(url)
        artigo_web.download()
        artigo_web.parse()
        return artigo_web.title, artigo_web.text
    except Exception as e:
        print(f" Erro ao tentar ler o link: {e}")
        return None, None

def traduzir_texto_azure(texto, idioma_destino='pt'):
    """Fatia o texto e usa Backoff Exponencial para vencer o erro 429."""
    url = TRANSLATOR_ENDPOINT + '/translate'
    params = {'api-version': '3.0', 'to': idioma_destino}
    headers = {
        'Ocp-Apim-Subscription-Key': TRANSLATOR_CHAVE,
        'Ocp-Apim-Subscription-Region': TRANSLATOR_REGIAO,
        'Content-type': 'application/json',
        'X-ClientTraceId': str(uuid.uuid4())
    }

    # Blocos ainda menores para não chamar atenção do filtro
    limite = 4000
    pedacos = []
    texto_restante = texto

    while len(texto_restante) > limite:
        corte = texto_restante.rfind('\n', 0, limite)
        if corte == -1: corte = texto_restante.rfind(' ', 0, limite)
        if corte == -1: corte = limite
        pedacos.append(texto_restante[:corte])
        texto_restante = texto_restante[corte:]
    pedacos.append(texto_restante)

    texto_traduzido_completo = ""
    total = len(pedacos)
    print(f"Dividido em {total} partes. Usando modo de segurança (lentidão proposital).")

    for i, pedaco in enumerate(pedacos):
        tentativas = 0
        sucesso = False
        espera_atual = 10 # Começa esperando 10 segundos

        while tentativas < 5 and not sucesso:
            print(f"   -> Parte {i+1}/{total} (Tentativa {tentativas + 1})...")
            resposta = requests.post(url, params=params, headers=headers, json=[{'text': pedaco}])

            if resposta.status_code == 200:
                dados_json = resposta.json()[0]

                # Exibe o idioma detectado apenas na primeira parte para não poluir o terminal
                if i == 0 and 'detectedLanguage' in dados_json:
                    idioma_origem = dados_json['detectedLanguage']['language']
                    score = dados_json['detectedLanguage']['score']
                    print(f"    Idioma detectado: {idioma_origem} (Confiança: {score*100}%)")

                texto_traduzido_completo += dados_json['translations'][0]['text'] + " "
                sucesso = True
            elif resposta.status_code == 429:
                print(f"        Azure cansou. Esperando {espera_atual}s para recuperar...")
                time.sleep(espera_atual)
                espera_atual *= 2 # Dobra o tempo de espera (Backoff Exponencial)
                tentativas += 1
            else:
                print(f"       Erro crítico: {resposta.text}")
                return None

        if not sucesso:
            print(" Falha após várias tentativas. A Azure bloqueou o IP temporariamente.")
            return None

        # Pausa fixa de segurança entre partes bem-sucedidas
        if i < total - 1:
            time.sleep(5)

    return texto_traduzido_completo

def formatar_com_ia_gratis(titulo, texto_traduzido):
    prompt = f"""
    Atue como um editor especialista em Markdown.

    Regras:
    1. Crie um Título Principal (H1) atrativo em português baseado no título original: '{titulo}'.
    2. Formate o texto abaixo de forma fluida, colocando conceitos chave em **negrito** e dividindo em parágrafos.
    3. Retorne APENAS o código Markdown em português, sem conversinha. NÃO adicione o texto original.

    Texto para formatar:
    {texto_traduzido}
    """

    print("Pedindo para a IA formatar a tradução (Isso pode levar alguns segundos)...")
    try:
        resposta = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt
        )
        texto_limpo = resposta.text.replace("```markdown", "").replace("```", "").strip()
        return texto_limpo
    except Exception as e:
        print(f"Erro na IA: {e}")
        return None

def salvar_arquivo(conteudo_md, nome_do_arquivo):
    if conteudo_md:
        with open(f"{nome_do_arquivo}.md", "w", encoding="utf-8") as arquivo:
            arquivo.write(conteudo_md)
        print(f" Sucesso! '{nome_do_arquivo}.md' gerado com sucesso.")

# ==========================================
# FLUXO PRINCIPAL
# ==========================================
if __name__ == "__main__":

    parser = argparse.ArgumentParser(description="Tradutor automático de artigos da web para Markdown.")
    parser.add_argument("-u", "--url", required=True, help="O link (URL) do artigo que você quer traduzir.")

    args = parser.parse_args()
    link_do_artigo = args.url

    print("Iniciando o processo...")

    titulo, artigo = extrair_artigo_da_url(link_do_artigo)

    if titulo and artigo:
        print(f"\nTítulo encontrado: {titulo}")
        print("1. Iniciando comunicação com a Azure...")
        texto_pt = traduzir_texto_azure(artigo, 'pt')

        if texto_pt:
            print("2. Formatando com IA Gratuita...")
            markdown_ia = formatar_com_ia_gratis(titulo, texto_pt)

            if markdown_ia:
                print("3. Montando o documento final via Python...")
                citacao_original = "> " + artigo.replace("\n", "\n> ")
                markdown_final = f"{markdown_ia}\n\n---\n\n## Texto Original\n\n{citacao_original}"

                print("4. Salvando...")
                nome_seguro = "".join([c if c.isalnum() else "_" for c in titulo])[:30]
                salvar_arquivo(markdown_final, f"Artigo_{nome_seguro}")
    else:
        print("Não foi possível continuar porque o texto não foi extraído do link.")

Overwriting translate.py


###  Execução e Resultados

Agora que o ambiente está preparado e o script salvo, veja como gerar e acessar seus documentos.

#### **Como Executar**
Utilize o comando abaixo, substituindo a URL entre aspas pelo artigo que você deseja traduzir:

```bash
!python translate.py -u "SUA_URL_AQUI"
```

#### **Como Baixar o Arquivo**
1. No menu lateral esquerdo do Google Colab, clique no ícone de **Pasta** (Arquivos).
2. Localize o arquivo com a extensão `.md` (ex: `Artigo_Titulo.md`).
3. Clique nos três pontos ao lado do arquivo e selecione **Fazer download**.

#### **Como Visualizar o Markdown**
O formato Markdown (`.md`) é um padrão de documentação técnica. Para visualizá-lo com a formatação (negritos, títulos e listas) que a IA aplicou, você pode usar:

*   **VS Code:** (Recomendado) Abra o arquivo e aperte `Ctrl + Shift + V`.
*   **Obsidian:** Excelente para organizar sua base de conhecimento.
*   **Visualizadores Online:** Como o [StackEdit](https://stackedit.io/) ou [Dillinger](https://dillinger.io/).

In [40]:
!python translate.py -u "https://zh.wikipedia.org/wiki/人工智能"

Iniciando o processo...
Lendo o link: https://zh.wikipedia.org/wiki/人工智能...
Building prefix dict from /usr/local/lib/python3.12/dist-packages/jieba/dict.txt ...
Loading model from cache /tmp/jieba.cache
Loading model cost 1.3583166599273682 seconds.
Prefix dict has been built succesfully.

Título encontrado: 维基百科，自由的百科全书
1. Iniciando comunicação com a Azure...
Dividido em 4 partes. Usando modo de segurança (lentidão proposital).
   -> Parte 1/4 (Tentativa 1)...
    Idioma detectado: zh-Hans (Confiança: 99.0%)
   -> Parte 2/4 (Tentativa 1)...
   -> Parte 3/4 (Tentativa 1)...
   -> Parte 4/4 (Tentativa 1)...
2. Formatando com IA Gratuita...
Pedindo para a IA formatar a tradução (Isso pode levar alguns segundos)...
3. Montando o documento final via Python...
4. Salvando...
 Sucesso! 'Artigo_维基百科_自由的百科全书.md' gerado com sucesso.
